# Lab 18.4: When the Monitor Becomes the Reward

Chapter 18, Steering, Editing and Monitoring Models. Run-only notebook in plain numpy (a few seconds), with one cell you may edit. This is a toy, not a reproduction of any experiment: a policy chooses among three strategies on coding tasks, trained by exact policy gradient. It shows the mechanism behind the obfuscation result of Baker et al. (2025) in a form you can take apart.

- **solve**: try honestly; passes the tests with probability 0.55.
- **hack, visible**: exploit the tests and write the plan in the trace; always passes; the trace monitor flags it with probability 0.95.
- **hack, hidden**: exploit the tests and write an innocent-looking trace; passes with probability 0.97 (hiding costs a little); flagged with probability 0.05.

In [1]:
import numpy as np
def train(penalty=0.0, flag=(0.0, 0.95, 0.05), task=(0.55, 1.0, 0.97), init=(0.0, 0.3, -3.0), steps=1500, lr=0.4):
    flag, task, th = np.array(flag), np.array(task), np.array(init, dtype=float)
    hist = []
    for _ in range(steps):
        pi = np.exp(th - th.max()); pi /= pi.sum()
        R = task - penalty * flag                      # the monitor enters the reward only if penalty > 0
        th += lr * pi * (R - pi @ R)                   # exact policy gradient for a softmax policy
        hack = pi[1] + pi[2]
        hist.append((pi[0], hack, (pi[1] * flag[1] + pi[2] * flag[2]) / hack))
    return np.array(hist)
def show(h, label):
    for s in (0, 100, 300, 600, 900, 1499):
        print(f"{label:22s} step {s:4d}: solve {h[s,0]:.2f}  hack {h[s,1]:.2f}  monitor catches {h[s,2]:.2f} of hacks")

## Step 1: watch only versus penalize

**Checkpoint 1:** when the monitor only watches, the policy learns to hack openly and the monitor catches about 95 percent of the hacks: bad behavior, but visible. When the monitor's flag is subtracted from the reward, honest solving rises first (the low-pressure benefit), then the hidden hack takes over: by the end the policy hacks in nearly every episode and the monitor catches about 5 percent of hacks.

In [2]:
show(train(0.0), "watch only")
print()
show(train(1.0), "monitor in reward")

watch only             step    0: solve 0.42  hack 0.58  monitor catches 0.92 of hacks
watch only             step  100: solve 0.03  hack 0.97  monitor catches 0.94 of hacks
watch only             step  300: solve 0.01  hack 0.99  monitor catches 0.95 of hacks
watch only             step  600: solve 0.00  hack 1.00  monitor catches 0.95 of hacks
watch only             step  900: solve 0.00  hack 1.00  monitor catches 0.95 of hacks
watch only             step 1499: solve 0.00  hack 1.00  monitor catches 0.95 of hacks

monitor in reward      step    0: solve 0.42  hack 0.58  monitor catches 0.92 of hacks
monitor in reward      step  100: solve 0.96  hack 0.04  monitor catches 0.72 of hacks
monitor in reward      step  300: solve 0.98  hack 0.02  monitor catches 0.51 of hacks
monitor in reward      step  600: solve 0.97  hack 0.03  monitor catches 0.27 of hacks
monitor in reward      step  900: solve 0.03  hack 0.97  monitor catches 0.05 of hacks
monitor in reward      step 1499: solve 0.

## Step 2: how much pressure?

**Checkpoint 2:** every penalty from 0.25 to 2 ends with the hidden hack; the penalty changes only how long honest solving lasts first (the hidden hack passes 50 percent of episodes after roughly 200 to 2,800 steps). A penalty of 4 delays the takeover past the 8,000-step budget, although hiding still pays more than honest work, so longer training would get there. Lowering the chance that a hack passes the tests far enough (to 0.6 here, where even a hidden hack pays less than honest work) removes the incentive to hack altogether; at 0.8 it does not. That is a better task check, not a stronger monitor.

In [3]:
print("penalty  final solve  final hack  catches  step where hidden hack passes 50%")
for p in (0.0, 0.25, 0.5, 1.0, 2.0, 4.0):
    h = train(p, steps=8000)
    hidden = h[:, 1] * (1 - (h[:, 2] - 0.05) / 0.90)     # share of episodes with a hidden hack
    t = int(np.argmax(hidden > 0.5)) if (hidden > 0.5).any() else "not within 8,000 steps"
    print(f"{p:7.2f} {h[-1,0]:12.2f} {h[-1,1]:11.2f} {h[-1,2]:8.2f}  {t}")
print()
print("hack pass rate  final solve  final hack")
for pr in (1.0, 0.8, 0.6, 0.4):
    h = train(1.0, task=(0.55, pr, pr - 0.03), steps=8000)
    print(f"{pr:14.2f} {h[-1,0]:12.2f} {h[-1,1]:11.2f}")

penalty  final solve  final hack  catches  step where hidden hack passes 50%
   0.00         0.00        1.00     0.95  not within 8,000 steps
   0.25         0.00        1.00     0.05  1351
   0.50         0.00        1.00     0.05  226


   1.00         0.00        1.00     0.05  785
   2.00         0.00        1.00     0.05  2814
   4.00         1.00        0.00     0.09  not within 8,000 steps

hack pass rate  final solve  final hack
          1.00         0.00        1.00
          0.80         0.00        1.00


          0.60         1.00        0.00
          0.40         1.00        0.00


## Your turn

Change one assumption and predict the result before you run it: a monitor that also catches hidden hacks half of the time (`flag=(0.0, 0.95, 0.5)`), a larger cost of hiding (`task=(0.55, 1.0, 0.8)`), or a policy that starts out hiding more readily (`init=(0.0, 0.3, -1.0)`).

In [4]:
h = train(1.0, flag=(0.0, 0.95, 0.5))   # edit me
show(h, "your setting")

your setting           step    0: solve 0.42  hack 0.58  monitor catches 0.93 of hacks
your setting           step  100: solve 0.96  hack 0.04  monitor catches 0.86 of hacks
your setting           step  300: solve 0.99  hack 0.01  monitor catches 0.81 of hacks
your setting           step  600: solve 0.99  hack 0.01  monitor catches 0.78 of hacks
your setting           step  900: solve 1.00  hack 0.00  monitor catches 0.76 of hacks
your setting           step 1499: solve 1.00  hack 0.00  monitor catches 0.74 of hacks


## Questions

1. In Step 1, at which step would you stop training if you could see only the monitor's flag rate? What would you have shipped?
2. Write two sentences for a model risk committee explaining why "fewer flags after training" is not evidence of fewer problems.
3. Which change in Step 2 removed the incentive to hack instead of the incentive to hide? What is its real-world counterpart?